# Lesson 7 — Van't Kruijs Opening 1.e3 (as Black)
**Project 1500 | Priority #7**

All statistics computed fresh from game files on every run. No hardcoded results.
Board diagrams are illustrative example lines, clearly labelled as such.

> **Sample size warning:** Very few games in this opening. Stats are directional only.

In [ ]:
import chess
import chess.pgn
import chess.svg
import json
import glob
import io
import re
from collections import Counter
from IPython.display import SVG, display, HTML
import matplotlib.pyplot as plt

USERNAME   = 'jf4bes'
EXCLUDE    = {'2025_01', '2025_02', '2025_03'}
BOARD_SIZE = 380

def board_after(moves_san):
    """Apply SAN moves to a fresh board — illustrative diagrams only."""
    board = chess.Board()
    for san in moves_san:
        board.push_san(san)
    return board

def show_board(board, arrows=None, caption='', size=BOARD_SIZE, flipped=True):
    svg = chess.svg.board(board, arrows=arrows or [], size=size, flipped=flipped)
    if caption:
        display(HTML(f'<p style="font-family:monospace;font-size:13px;margin:4px 0 8px 0;color:#aaa">{caption}</p>'))
    display(SVG(data=svg))

## Step 1 — Load rapid games as black (April 2025+)

In [ ]:
def load_rapid_games_as_black(username, games_dir='games', exclude=EXCLUDE):
    files = sorted(glob.glob(f'{games_dir}/2025_*.json') + glob.glob(f'{games_dir}/2026_*.json'))
    files = [f for f in files if not any(ex in f for ex in exclude)]
    games = []
    for f in files:
        with open(f, encoding='utf-8') as fh:
            month = json.load(fh)
        for g in month:
            if g.get('time_class') != 'rapid': continue
            black = g.get('black', {})
            if black.get('username', '').lower() != username.lower(): continue
            result = black.get('result', '')
            if   result == 'win': outcome = 'win'
            elif result in ('checkmated','timeout','resigned','lose','abandoned'): outcome = 'loss'
            elif result in ('agreed','stalemate','repetition','insufficient','timevsinsufficient','50move'): outcome = 'draw'
            else: continue
            eco_url = ''
            m = re.search(r'\[ECOUrl "([^"]+)"\]', g.get('pgn', ''))
            if m: eco_url = m.group(1)
            games.append({'outcome': outcome, 'pgn': g.get('pgn', ''), 'eco_url': eco_url})
    return games

all_black_games = load_rapid_games_as_black(USERNAME)
print(f'Loaded {len(all_black_games)} rapid games as black')

## Step 2 — Detect Van't Kruijs games (white's first move is e3)

Detected by position: after white's first move, the pawn is on e3 and nowhere else moved.

In [ ]:
vant_kruijs_games = []

for g in all_black_games:
    try:
        game  = chess.pgn.read_game(io.StringIO(g['pgn']))
        board = game.board()
        moves = list(game.mainline_moves())
        if not moves: continue
        # Check white's very first move
        first_san = board.san(moves[0])
        if first_san == 'e3':
            vant_kruijs_games.append(g)
    except Exception:
        pass

counts = Counter(g['outcome'] for g in vant_kruijs_games)
total  = len(vant_kruijs_games)
wr     = 100 * counts['win'] / total if total else 0

print(f"Van't Kruijs games as black: {total}")
print(f'W:{counts["win"]}  L:{counts["loss"]}  D:{counts["draw"]}  Win%: {wr:.1f}%')

if total < 10:
    print('\nSmall sample — the lesson here is principle-based: you should be winning these.')
    print('Any loss to 1.e3 is a problem worth understanding.')

## Step 3 — Board diagrams *(illustrative example lines)*

With a small sample we focus on teaching the correct principled response rather than statistics.

In [ ]:
# [ILLUSTRATIVE] After 1.e3 — seize the centre immediately
# Verified legal: e3 (white's move), black responds
AFTER_E3 = ['e3']

show_board(
    board_after(AFTER_E3),
    arrows=[
        chess.svg.Arrow(chess.D7, chess.D5, color='#27ae60'),  # d5 — seize centre
        chess.svg.Arrow(chess.E7, chess.E5, color='#27ae60'),  # e5 — also good
    ],
    caption='[ILLUSTRATIVE] After 1.e3 — play d5 or e5 immediately. White has wasted a move. Claim the centre.'
)

In [ ]:
# [ILLUSTRATIVE] Black's ideal setup against the Van't Kruijs
# Verified legal: e3 d5 d4 Nf6 Nf3 Bf5 Be2 e6 O-O Bd6
IDEAL_SETUP = ['e3','d5','d4','Nf6','Nf3','Bf5','Be2','e6','O-O','Bd6']

show_board(
    board_after(IDEAL_SETUP),
    arrows=[
        chess.svg.Arrow(chess.F5, chess.B1, color='#27ae60'),  # Bf5 active
        chess.svg.Arrow(chess.D6, chess.H2, color='#2980b9'),  # Bd6 eyes h2
    ],
    caption='[ILLUSTRATIVE] Black\'s ideal setup — both bishops active, d5 controls the centre, castled safely'
)

In [ ]:
# [ILLUSTRATIVE] The Qh5 trap that appears in your losses — same fix as Lesson 6
# Verified legal: e3 e5 Qh5 Nc6 (black's correct response)
QH5_TRAP = ['e3','e5','Qh5']

show_board(
    board_after(QH5_TRAP),
    arrows=[
        chess.svg.Arrow(chess.B8, chess.C6, color='#27ae60'),  # Nc6 correct
        chess.svg.Arrow(chess.G7, chess.G6, color='#c0392b'),  # g6 blunder
    ],
    caption='[ILLUSTRATIVE] Qh5 trap appears here too — same fix: Nc6 (green), never g6 (red)'
)

## Step 4 — Summary

In [ ]:
print("VAN'T KRUIJS — SUMMARY")
print('=' * 50)
print(f"\nTotal Van't Kruijs games as black: {total}  ({wr:.1f}% win rate)")
if total < 10:
    print('(Small sample — focus on the principle)')
print()
print('RULES:')
print('  1. Play d5 or e5 immediately — claim the centre, white wasted a move')
print('  2. Develop actively: Nf6, Bf5 (outside the pawn chain), Bd6')
print('  3. Castle kingside')
print('  4. If white plays Qh5 — Nc6 (same rule as Lesson 6, never g6)')
print('  5. Push c5 or e5 to punish white\'s passive opening')